# KuaiRand Feature Analysis and Selection

Goal: decide a compact but useful feature set for the adaptive recommender gold layer.

This notebook is intentionally not a general EDA notebook. It focuses on:

- point-in-time feature candidates
- leakage review before feature importance
- statistical evidence and redundancy
- temporal stability
- segment behavior, especially high-drift sessions
- Spark ML model importance and group ablation

The silver layer may keep many fields. Feature selection here is for gold/model design, so the final recommendation can be smaller than the available silver schema.


## 1. Setup

All heavy data access stays in Spark. Small summary tables may be converted to pandas only after aggregation.


In [1]:
from pathlib import Path
import math
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'recommender').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, VectorAssembler

from recommender.spark import get_spark

spark = get_spark('kuairand-feature-analysis', reset=True)
spark.sparkContext.setLogLevel('WARN')

SILVER_DIR = PROJECT_ROOT / 'data/silver/kuairand'
TARGET_COL = 'target_long_view'
SEED = 42
SAMPLE_MODULO = 1000
SAMPLE_BUCKETS = 20  # deterministic ~2% sample; adjust upward on a larger machine
MAX_MODEL_ROWS = 250_000

print(f'Project root: {PROJECT_ROOT}')
print(f'Silver dir: {SILVER_DIR}')
print(f'Spark {spark.version} | master={spark.sparkContext.master}')


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 17:02:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 17:02:01 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Project root: /Users/khoatran/coding/recsys
Silver dir: /Users/khoatran/coding/recsys/data/silver/kuairand
Spark 3.5.6 | master=local[*]


## 2. Load and Inspect Silver Tables

The notebook does not assume a fixed schema. It reads the silver tables that exist, prints schemas, and derives candidate columns from actual data types.


In [2]:
required_tables = ['interactions', 'users', 'videos_basic', 'videos_statistics']
missing = [name for name in required_tables if not (SILVER_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f'Missing silver tables: {missing}. Run: python scripts/build_kuairand_silver.py --overwrite'
    )

tables = {name: spark.read.parquet(str(SILVER_DIR / name)) for name in required_tables}
for name, df in tables.items():
    print(f'\n{name}: rows={df.count():,}, columns={len(df.columns)}')
    df.printSchema()



interactions: rows=11,756,073, columns=31
root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- video_id: integer (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_minute: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- hourmin: integer (nullable = true)
 |-- time_ms: long (nullable = true)
 |-- is_click: integer (nullable = true)
 |-- is_like: integer (nullable = true)
 |-- is_follow: integer (nullable = true)
 |-- is_comment: integer (nullable = true)
 |-- is_forward: integer (nullable = true)
 |-- is_hate: integer (nullable = true)
 |-- long_view: integer (nullable = true)
 |-- is_profile_enter: integer (nullable = true)
 |-- play_time_ms: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- play_time_sec: double (nullable = true)
 |-- duration_sec: double (nullable = true)
 |-- watch_ratio: double (nu

In [3]:
schema_rows = []
for table_name, df in tables.items():
    for field in df.schema.fields:
        dtype = field.dataType.simpleString()
        if isinstance(field.dataType, (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)):
            role = 'numeric'
        elif isinstance(field.dataType, (T.DateType, T.TimestampType)):
            role = 'temporal'
        else:
            role = 'categorical_or_text'
        schema_rows.append((table_name, field.name, dtype, role))

schema_df = spark.createDataFrame(schema_rows, ['table', 'column', 'dtype', 'inferred_role'])
schema_df.orderBy('table', 'column').show(200, truncate=False)


+-----------------+-------------------------+---------+-------------------+
|table            |column                   |dtype    |inferred_role      |
+-----------------+-------------------------+---------+-------------------+
|interactions     |comment_stay_time        |int      |numeric            |
|interactions     |date                     |int      |numeric            |
|interactions     |duration_ms              |int      |numeric            |
|interactions     |duration_sec             |double   |numeric            |
|interactions     |event_date               |date     |temporal           |
|interactions     |event_hour               |int      |numeric            |
|interactions     |event_id                 |string   |categorical_or_text|
|interactions     |event_minute             |int      |numeric            |
|interactions     |event_ts                 |timestamp|temporal           |
|interactions     |hourmin                  |int      |numeric            |
|interaction

## 3. Feature Catalog Draft

This catalog is the design document inside the notebook. It includes raw fields, derived point-in-time aggregates, leakage risk, and serving/update assumptions.

Important: post-interaction columns such as `play_time_sec` and `watch_ratio` are useful for defining labels or offline diagnostics, but they are not available before recommending an item. They are flagged as leakage for ranking prediction.


In [4]:
def has_col(table_name, col):
    return col in tables[table_name].columns

def source_exists(sources):
    for source in sources:
        table, cols = source.split(':', 1)
        for col in cols.split(','):
            if not has_col(table, col.strip()):
                return False
    return True

catalog_specs = [
    ('user_hist_events', 'long_term_user', ['interactions:user_id,event_ts'], 'count prior events for user before prediction event', 'online', 'before event_ts', 'LOW', 'per interaction', 'Heavy users often have more stable preference estimates.'),
    ('user_hist_long_view_rate', 'long_term_user', ['interactions:user_id,long_view,event_ts'], 'prior long_view mean by user', 'online', 'before event_ts', 'LOW', 'per interaction', 'Captures long-term engagement tendency.'),
    ('user_hist_like_rate', 'long_term_user', ['interactions:user_id,is_like,event_ts'], 'prior like mean by user', 'online', 'before event_ts', 'LOW', 'per interaction', 'Like behavior is a stronger preference signal than exposure alone.'),
    ('user_hist_hate_rate', 'long_term_user', ['interactions:user_id,is_hate,event_ts'], 'prior hate mean by user', 'online', 'before event_ts', 'LOW', 'per interaction', 'Negative feedback can suppress incompatible recommendations.'),
    ('user_hist_avg_watch_ratio', 'long_term_user', ['interactions:user_id,watch_ratio,event_ts'], 'prior average watch_ratio by user', 'online', 'before event_ts', 'LOW', 'per interaction', 'Normalizes preference by video length.'),
    ('user_active_degree', 'long_term_user', ['users:user_active_degree'], 'raw user profile categorical feature', 'batch', 'profile snapshot', 'MEDIUM', 'daily or snapshot refresh', 'Activity segment may change recommendation strategy.'),
    ('follow_user_num', 'long_term_user', ['users:follow_user_num'], 'raw user profile count', 'batch', 'profile snapshot', 'MEDIUM', 'daily or snapshot refresh', 'Social graph size may proxy taste breadth.'),
    ('fans_user_num', 'long_term_user', ['users:fans_user_num'], 'raw user profile count', 'batch', 'profile snapshot', 'MEDIUM', 'daily or snapshot refresh', 'Creator/social status can correlate with behavior.'),
    ('register_days', 'long_term_user', ['users:register_days'], 'raw account age', 'batch', 'profile snapshot', 'MEDIUM', 'daily', 'Newer and older accounts may behave differently.'),
    ('session_event_index', 'session', ['interactions:user_id,event_ts'], 'number of prior sampled events in current 30-minute session', 'online', 'before event_ts', 'LOW', 'per interaction', 'Later session positions often reflect narrower short-term intent.'),
    ('session_elapsed_sec', 'session', ['interactions:user_id,event_ts'], 'seconds from session start to current event', 'online', 'before event_ts', 'LOW', 'per interaction', 'Session maturity can change exploration/exploitation balance.'),
    ('session_prior_long_view_rate', 'session', ['interactions:user_id,long_view,event_ts'], 'prior long_view rate within current session', 'online', 'before event_ts', 'LOW', 'per interaction', 'Captures immediate satisfaction in the session.'),
    ('session_prior_avg_watch_ratio', 'session', ['interactions:user_id,watch_ratio,event_ts'], 'prior average watch_ratio within current session', 'online', 'before event_ts', 'LOW', 'per interaction', 'Short-term intensity signal.'),
    ('session_vs_user_long_view_delta', 'drift/adaptation', ['interactions:user_id,long_view,event_ts'], 'session_prior_long_view_rate minus user_hist_long_view_rate', 'online', 'before event_ts', 'LOW', 'per interaction', 'Large deltas indicate current intent differs from long-term baseline.'),
    ('session_vs_user_watch_ratio_delta', 'drift/adaptation', ['interactions:user_id,watch_ratio,event_ts'], 'session_prior_avg_watch_ratio minus user_hist_avg_watch_ratio', 'online', 'before event_ts', 'LOW', 'per interaction', 'Measures preference drift in engagement depth.'),
    ('item_hist_events', 'item', ['interactions:video_id,event_ts'], 'count prior sampled events for item before event_ts', 'online/batch', 'before event_ts', 'LOW', 'near real time', 'Prior exposure volume improves cold/popular item handling.'),
    ('item_hist_long_view_rate', 'item', ['interactions:video_id,long_view,event_ts'], 'prior item long_view rate before event_ts', 'online/batch', 'before event_ts', 'LOW', 'near real time', 'Point-in-time item quality signal.'),
    ('video_duration_sec', 'item', ['videos_basic:video_duration_sec'], 'duration from video metadata', 'batch', 'before serving if metadata snapshot is current', 'LOW', 'on item update', 'Video length affects completion and long-view probability.'),
    ('aspect_ratio', 'item', ['videos_basic:aspect_ratio'], 'server_width / server_height', 'batch', 'before serving if metadata snapshot is current', 'LOW', 'on item update', 'Format can affect engagement.'),
    ('video_type', 'item', ['videos_basic:video_type'], 'raw video type categorical metadata', 'batch', 'before serving if metadata snapshot is current', 'LOW', 'on item update', 'Content format is a coarse preference signal.'),
    ('music_type', 'item', ['videos_basic:music_type'], 'raw music type categorical metadata', 'batch', 'before serving if metadata snapshot is current', 'LOW', 'on item update', 'Audio context can affect short-video preference.'),
    ('show_cnt', 'item', ['videos_statistics:show_cnt'], 'global historical show count from provided statistic table', 'batch', 'snapshot time unknown', 'HIGH', 'snapshot refresh', 'Popularity can be strong but must be point-in-time.'),
    ('play_per_show', 'item', ['videos_statistics:play_per_show'], 'play_cnt / show_cnt from provided statistic table', 'batch', 'snapshot time unknown', 'HIGH', 'snapshot refresh', 'Quality/popularity ratio; risky if computed over future period.'),
    ('like_rate', 'item', ['videos_statistics:like_rate'], 'like_cnt / play_cnt from provided statistic table', 'batch', 'snapshot time unknown', 'HIGH', 'snapshot refresh', 'Quality signal; risky without timestamped statistic window.'),
    ('comment_rate', 'item', ['videos_statistics:comment_rate'], 'comment_cnt / play_cnt from provided statistic table', 'batch', 'snapshot time unknown', 'HIGH', 'snapshot refresh', 'Deep engagement signal; risky without timestamped statistic window.'),
    ('event_hour', 'context', ['interactions:event_hour'], 'hour extracted from event timestamp', 'online', 'at request time', 'LOW', 'per request', 'Time-of-day preference can be meaningful.'),
    ('event_dayofweek', 'context', ['interactions:event_ts'], 'day of week from event timestamp', 'online', 'at request time', 'LOW', 'per request', 'Weekday/weekend behavior may differ.'),
    ('is_rand', 'context', ['interactions:is_rand'], 'random traffic flag', 'offline/evaluation', 'known in log', 'MEDIUM', 'per event', 'Useful for analysis; serving availability depends on traffic source.'),
    ('tab', 'context', ['interactions:tab'], 'tab/source context from log', 'online', 'at request time if serving context has tab', 'LOW', 'per request', 'Different surfaces may need different ranking behavior.'),
    ('play_time_sec', 'context', ['interactions:play_time_sec'], 'post-event play time', 'offline only', 'after interaction', 'HIGH', 'after event', 'Useful label/debug signal, not a pre-ranking feature.'),
    ('watch_ratio', 'context', ['interactions:watch_ratio'], 'post-event play_time / duration', 'offline only', 'after interaction', 'HIGH', 'after event', 'Useful label/debug signal, not a pre-ranking feature.'),
]

catalog_rows = []
for name, group, sources, formula, compute_mode, available_at, leakage, update_frequency, hypothesis in catalog_specs:
    catalog_rows.append({
        'feature': name,
        'group': group,
        'source_columns': '; '.join(sources),
        'derivation': formula,
        'batch_or_online': compute_mode,
        'available_at_prediction_time': available_at,
        'leakage_risk': leakage,
        'expected_update_frequency': update_frequency,
        'hypothesis': hypothesis,
        'exists_in_current_data': source_exists(sources),
    })

feature_catalog = spark.createDataFrame(catalog_rows)
feature_catalog.orderBy('group', 'feature').show(80, truncate=False)


+----------------------------------------------+------------------+-------------------------------------------------------------+----------------------+-------------------------+---------------------------------+----------------+---------------------------------------------------------------------+------------+-----------------------------------------+
|available_at_prediction_time                  |batch_or_online   |derivation                                                   |exists_in_current_data|expected_update_frequency|feature                          |group           |hypothesis                                                           |leakage_risk|source_columns                           |
+----------------------------------------------+------------------+-------------------------------------------------------------+----------------------+-------------------------+---------------------------------+----------------+-------------------------------------------------------------

## 4. Build a Point-in-Time Analysis Dataset

For expensive statistical and model-based checks we use a deterministic event sample. Derived history features use only rows before the prediction event within that sample. In production/gold, the same formulas should run over the full stream or feature store, not the sample.


In [5]:
interactions = tables['interactions']
users = tables['users']
videos_basic = tables['videos_basic']
videos_statistics = tables['videos_statistics']

base = (
    interactions
    .where(F.col('event_ts').isNotNull())
    .withColumn(TARGET_COL, F.coalesce(F.col('long_view'), F.lit(0)).cast('int'))
    .withColumn('event_dayofweek', F.dayofweek('event_ts'))
    .withColumn('sample_bucket', F.pmod(F.abs(F.xxhash64('event_id')), F.lit(SAMPLE_MODULO)))
    .where(F.col('sample_bucket') < SAMPLE_BUCKETS)
    .drop('sample_bucket')
)

sample_count = base.count()
print(f'Deterministic event sample rows: {sample_count:,}')
base.select('event_id', 'user_id', 'video_id', 'event_ts', TARGET_COL, 'event_hour', 'event_dayofweek').show(5, truncate=False)


Deterministic event sample rows: 235,658


+----------------------------------------------------------------+-------+--------+-------------------+----------------+----------+---------------+
|event_id                                                        |user_id|video_id|event_ts           |target_long_view|event_hour|event_dayofweek|
+----------------------------------------------------------------+-------+--------+-------------------+----------------+----------+---------------+
|e0639527cdd257fb6f6d41b1ab802a7b129887a2dbfe0dfb9769bcb254c6f639|0      |2187381 |2022-04-22 00:01:53|1               |8         |6              |
|51fb70c4651c220af6d61fb074083e37088bcfc9a124620c1b05a89072696e0e|0      |3598175 |2022-04-22 00:16:20|0               |8         |6              |
|be4e443dce285f1fb26fb92e34328d50fbf36d59338dabeeab475151f6f1b91f|0      |1518926 |2022-04-22 00:21:24|0               |8         |6              |
|d194589fb413b1fe5196e953b587e0072eea77f82b132a20fc95e1fa23341bd6|0      |3540796 |2022-04-22 00:21:45|1        

In [6]:
w_user_order = Window.partitionBy('user_id').orderBy('event_ts', 'event_id')
w_user_prev = w_user_order.rowsBetween(Window.unboundedPreceding, -1)
w_user_cur = w_user_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)

with_user_history = (
    base
    .withColumn('prev_event_ts', F.lag('event_ts').over(w_user_order))
    .withColumn('gap_sec', F.col('event_ts').cast('long') - F.col('prev_event_ts').cast('long'))
    .withColumn('new_session_flag', F.when(F.col('prev_event_ts').isNull() | (F.col('gap_sec') > 30 * 60), 1).otherwise(0))
    .withColumn('session_seq', F.sum('new_session_flag').over(w_user_cur))
    .withColumn('session_id', F.concat_ws('_', F.col('user_id').cast('string'), F.col('session_seq').cast('string')))
    .withColumn('user_hist_events', F.count(F.lit(1)).over(w_user_prev))
    .withColumn('user_hist_long_view_rate', F.avg(F.col(TARGET_COL).cast('double')).over(w_user_prev))
    .withColumn('user_hist_like_rate', F.avg(F.coalesce('is_like', F.lit(0)).cast('double')).over(w_user_prev))
    .withColumn('user_hist_hate_rate', F.avg(F.coalesce('is_hate', F.lit(0)).cast('double')).over(w_user_prev))
    .withColumn('user_hist_avg_watch_ratio', F.avg('watch_ratio').over(w_user_prev))
)

w_session_order = Window.partitionBy('user_id', 'session_seq').orderBy('event_ts', 'event_id')
w_session_prev = w_session_order.rowsBetween(Window.unboundedPreceding, -1)
w_session_cur = w_session_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_item_order = Window.partitionBy('video_id').orderBy('event_ts', 'event_id')
w_item_prev = w_item_order.rowsBetween(Window.unboundedPreceding, -1)

with_history = (
    with_user_history
    .withColumn('session_event_index', F.count(F.lit(1)).over(w_session_prev))
    .withColumn('session_start_ts', F.first('event_ts').over(w_session_cur))
    .withColumn('session_elapsed_sec', F.col('event_ts').cast('long') - F.col('session_start_ts').cast('long'))
    .withColumn('session_prior_long_view_rate', F.avg(F.col(TARGET_COL).cast('double')).over(w_session_prev))
    .withColumn('session_prior_avg_watch_ratio', F.avg('watch_ratio').over(w_session_prev))
    .withColumn('item_hist_events', F.count(F.lit(1)).over(w_item_prev))
    .withColumn('item_hist_long_view_rate', F.avg(F.col(TARGET_COL).cast('double')).over(w_item_prev))
    .withColumn('session_vs_user_long_view_delta', F.col('session_prior_long_view_rate') - F.col('user_hist_long_view_rate'))
    .withColumn('session_vs_user_watch_ratio_delta', F.col('session_prior_avg_watch_ratio') - F.col('user_hist_avg_watch_ratio'))
)

user_profile_cols = [c for c in ['user_active_degree', 'follow_user_num', 'fans_user_num', 'register_days', 'friend_user_num'] if c in users.columns]
video_basic_cols = [c for c in ['video_type', 'upload_type', 'visible_status', 'video_duration_sec', 'aspect_ratio', 'music_type'] if c in videos_basic.columns]
video_stats_cols = [c for c in ['show_cnt', 'play_cnt', 'play_per_show', 'valid_play_rate', 'long_play_rate', 'like_rate', 'comment_rate', 'share_rate', 'follow_rate'] if c in videos_statistics.columns]

feature_df = (
    with_history
    .join(F.broadcast(users.select('user_id', *user_profile_cols)), on='user_id', how='left')
    .join(videos_basic.select('video_id', *video_basic_cols), on='video_id', how='left')
    .join(videos_statistics.select('video_id', *video_stats_cols), on='video_id', how='left')
)

# Temporal split: chronological train/validation/test by event timestamp.
q70, q85 = feature_df.approxQuantile('time_ms', [0.70, 0.85], 0.001)
feature_df = (
    feature_df
    .withColumn('split', F.when(F.col('time_ms') < q70, F.lit('train')).when(F.col('time_ms') < q85, F.lit('validation')).otherwise(F.lit('test')))
    .cache()
)

print(f'time_ms split cutoffs: q70={int(q70)}, q85={int(q85)}')
feature_df.groupBy('split').agg(F.count('*').alias('rows'), F.avg(TARGET_COL).alias('target_rate')).orderBy('split').show(truncate=False)


26/09/23 17:02:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


time_ms split cutoffs: q70=1651305565709, q85=1651655354081


+----------+------+------------------+
|split     |rows  |target_rate       |
+----------+------+------------------+
|test      |35543 |0.2550713220606026|
|train     |164784|0.2632355083017769|
|validation|35331 |0.2606492881605389|
+----------+------+------------------+



Interpretation: the split above is chronological, not random. All model and ablation conclusions below should be read as temporal validation/test evidence.


## 5. Basic Feature Quality

We check missingness, coverage, cardinality, variance, and outlier-ish ranges. This does not decide features alone; it catches fields that are impossible or expensive to use reliably.


In [7]:
existing_catalog = [r['feature'] for r in catalog_rows if r['exists_in_current_data']]
existing_features = [c for c in existing_catalog if c in feature_df.columns]

numeric_types = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
numeric_features = [f.name for f in feature_df.schema.fields if f.name in existing_features and isinstance(f.dataType, numeric_types)]
categorical_features = [f.name for f in feature_df.schema.fields if f.name in existing_features and not isinstance(f.dataType, numeric_types)]

print(f'Existing catalog features: {len(existing_features)}')
print(f'Numeric features available for statistics/modeling: {len(numeric_features)}')
print(f'Categorical/text features available for catalog/cardinality: {len(categorical_features)}')
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)


Existing catalog features: 31
Numeric features available for statistics/modeling: 29
Categorical/text features available for catalog/cardinality: 2
Numeric: ['event_hour', 'play_time_sec', 'watch_ratio', 'is_rand', 'tab', 'event_dayofweek', 'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate', 'user_hist_hate_rate', 'user_hist_avg_watch_ratio', 'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate', 'session_prior_avg_watch_ratio', 'item_hist_events', 'item_hist_long_view_rate', 'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta', 'follow_user_num', 'fans_user_num', 'register_days', 'video_duration_sec', 'aspect_ratio', 'music_type', 'show_cnt', 'play_per_show', 'like_rate', 'comment_rate']
Categorical: ['user_active_degree', 'video_type']


In [8]:
total_rows = feature_df.count()
quality_rows = []
for col in existing_features:
    dtype = feature_df.schema[col].dataType.simpleString()
    agg = feature_df.agg(
        F.count(F.col(col)).alias('non_null'),
        F.approx_count_distinct(F.col(col)).alias('approx_cardinality')
    ).first()
    row = {
        'feature': col,
        'dtype': dtype,
        'missing_rate': 1.0 - (agg['non_null'] / total_rows if total_rows else 0.0),
        'coverage': agg['non_null'] / total_rows if total_rows else 0.0,
        'approx_cardinality': int(agg['approx_cardinality'] or 0),
    }
    if col in numeric_features:
        stats = feature_df.select(F.col(col).cast('double').alias(col)).agg(
            F.mean(col).alias('mean'), F.stddev(col).alias('stddev'),
            F.min(col).alias('min'), F.expr(f'percentile_approx({col}, 0.5)').alias('median'), F.max(col).alias('max')
        ).first()
        row.update({k: (float(stats[k]) if stats[k] is not None else None) for k in ['mean', 'stddev', 'min', 'median', 'max']})
        row['near_zero_variance'] = stats['stddev'] is None or float(stats['stddev']) < 1e-9
    else:
        row.update({'mean': None, 'stddev': None, 'min': None, 'median': None, 'max': None, 'near_zero_variance': None})
    quality_rows.append(row)

quality_df = spark.createDataFrame(quality_rows)
quality_df.orderBy(F.desc('missing_rate'), 'feature').show(80, truncate=False)


+------------------+-------------------+------+---------------------------------+------------------+---------------------+--------------------+-------------------+--------------------+------------------+--------------------+
|approx_cardinality|coverage           |dtype |feature                          |max               |mean                 |median              |min                |missing_rate        |near_zero_variance|stddev              |
+------------------+-------------------+------+---------------------------------+------------------+---------------------+--------------------+-------------------+--------------------+------------------+--------------------+
|34                |0.10899693623810777|double|item_hist_long_view_rate         |1.0               |0.35203967068601505  |0.0                 |0.0                |0.8910030637618922  |false             |0.44089499602336174 |
|113319            |0.5910768995748076 |double|session_prior_avg_watch_ratio    |51.71027131782946 |

Interpretation: high missingness does not automatically mean DROP. For adaptive recommendation, sparse history features can still be useful if missingness corresponds to cold-start. They need default values and missing indicators in gold.


## 6. Leakage Review Before Importance

Features with `HIGH` leakage risk are not used in model-based importance or ablation below. They can remain in silver for labels/diagnostics but should not enter a pre-ranking model unless rebuilt point-in-time.


In [9]:
leakage_review = feature_catalog.select(
    'feature', 'group', 'source_columns', 'available_at_prediction_time', 'leakage_risk', 'derivation', 'exists_in_current_data'
).orderBy(F.desc('leakage_risk'), 'group', 'feature')
leakage_review.show(80, truncate=False)

safe_features = [
    r['feature'] for r in feature_catalog
    .where((F.col('exists_in_current_data') == True) & (F.col('leakage_risk') != 'HIGH'))
    .select('feature').collect()
    if r['feature'] in feature_df.columns
]
safe_numeric_features = [c for c in safe_features if c in numeric_features]
print('Safe numeric features for modeling:', safe_numeric_features)


+---------------------------------+----------------+-----------------------------------------+----------------------------------------------+------------+-------------------------------------------------------------+----------------------+
|feature                          |group           |source_columns                           |available_at_prediction_time                  |leakage_risk|derivation                                                   |exists_in_current_data|
+---------------------------------+----------------+-----------------------------------------+----------------------------------------------+------------+-------------------------------------------------------------+----------------------+
|is_rand                          |context         |interactions:is_rand                     |known in log                                  |MEDIUM      |random traffic flag                                          |true                  |
|fans_user_num                    |long_

Safe numeric features for modeling: ['user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate', 'user_hist_hate_rate', 'user_hist_avg_watch_ratio', 'follow_user_num', 'fans_user_num', 'register_days', 'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate', 'session_prior_avg_watch_ratio', 'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta', 'item_hist_events', 'item_hist_long_view_rate', 'video_duration_sec', 'aspect_ratio', 'music_type', 'event_hour', 'event_dayofweek', 'is_rand', 'tab']


## 7. Linear, Monotonic, and Mutual Information Signals

Pearson correlation is computed directly for continuous/numeric features. Spearman is approximated by correlating a feature decile rank with the binary target. Mutual information is estimated from binned Spark contingency tables, avoiding large one-hot matrices.


In [10]:
def pearson_corr(df, feature, target=TARGET_COL):
    clean = df.select(F.col(feature).cast('double').alias(feature), F.col(target).cast('double').alias(target)).na.drop()
    if clean.limit(2).count() < 2:
        return None
    try:
        value = clean.stat.corr(feature, target)
        return float(value) if value is not None and not math.isnan(value) else None
    except Exception:
        return None

def binned_mi(df, feature, target=TARGET_COL, buckets=10):
    clean = df.select(F.col(feature).cast('double').alias(feature), F.col(target).cast('int').alias(target)).na.drop()
    if clean.limit(2).count() < 2:
        return None
    qs = clean.approxQuantile(feature, [i / buckets for i in range(1, buckets)], 0.01)
    qs = sorted(set(float(q) for q in qs if q is not None))
    if not qs:
        return 0.0
    expr = F.lit(0)
    for idx, threshold in enumerate(qs):
        expr = F.when(F.col(feature) <= threshold, F.lit(idx)).otherwise(expr)
    # Build monotonically without accidentally making all non-first bins zero.
    bin_expr = F.lit(len(qs))
    for idx, threshold in reversed(list(enumerate(qs))):
        bin_expr = F.when(F.col(feature) <= threshold, F.lit(idx)).otherwise(bin_expr)
    counts = clean.withColumn('bin', bin_expr).groupBy('bin', target).count().cache()
    n = counts.agg(F.sum('count')).first()[0]
    if not n:
        return None
    px = counts.groupBy('bin').agg(F.sum('count').alias('cx'))
    py = counts.groupBy(target).agg(F.sum('count').alias('cy'))
    mi = (
        counts.join(px, 'bin').join(py, target)
        .select((F.col('count') / F.lit(n) * F.log((F.col('count') * F.lit(n)) / (F.col('cx') * F.col('cy')))).alias('term'))
        .agg(F.sum('term')).first()[0]
    )
    counts.unpersist()
    return float(mi or 0.0)

def decile_spearman_proxy(df, feature, target=TARGET_COL):
    clean = df.select(F.col(feature).cast('double').alias(feature), F.col(target).cast('double').alias(target)).na.drop()
    if clean.limit(2).count() < 2:
        return None
    qs = clean.approxQuantile(feature, [i / 10 for i in range(1, 10)], 0.01)
    qs = sorted(set(float(q) for q in qs if q is not None))
    if not qs:
        return 0.0
    bin_expr = F.lit(len(qs))
    for idx, threshold in reversed(list(enumerate(qs))):
        bin_expr = F.when(F.col(feature) <= threshold, F.lit(idx)).otherwise(bin_expr)
    ranked = clean.withColumn(f'{feature}_rank_bin', bin_expr.cast('double'))
    value = ranked.stat.corr(f'{feature}_rank_bin', target)
    return float(value) if value is not None and not math.isnan(value) else None

stat_signal_rows = []
for feature in safe_numeric_features:
    stat_signal_rows.append({
        'feature': feature,
        'pearson': pearson_corr(feature_df, feature),
        'spearman_proxy': decile_spearman_proxy(feature_df, feature),
        'mutual_information': binned_mi(feature_df, feature),
    })

stat_signal_df = spark.createDataFrame(stat_signal_rows)
stat_signal_df.orderBy(F.desc('mutual_information')).show(80, truncate=False)


+---------------------------------+---------------------+---------------------+---------------------+
|feature                          |mutual_information   |pearson              |spearman_proxy       |
+---------------------------------+---------------------+---------------------+---------------------+
|user_hist_long_view_rate         |0.05682218804661528  |0.3338613058274649   |0.3272704099683389   |
|user_hist_avg_watch_ratio        |0.04580675019169924  |0.26999444268306066  |0.2951382194602324   |
|tab                              |0.043297437574821754 |0.10722770450188732  |0.17946043917313068  |
|session_vs_user_long_view_delta  |0.028155524342992896 |0.07834829112508042  |0.0480935221228681   |
|session_prior_long_view_rate     |0.025106003631003064 |0.2270356325250238   |0.20294366476028117  |
|session_prior_avg_watch_ratio    |0.02389308327174799  |0.1532295995688679   |0.20442923272632937  |
|session_event_index              |0.016598797986734085 |-0.1375139986741855  |-0.

Interpretation: MI/correlation are screening tools. A feature with modest MI can still be useful in interactions with other features or in high-drift segments. A strong statistic can still be rejected if it leaks future information.


## 8. Redundancy Analysis

We check pairwise Pearson correlations among safe numeric candidates. Very high absolute correlation suggests duplicate signal; prefer the cheaper or more point-in-time reliable feature.


In [11]:
# Limit pairwise jobs to the strongest statistical candidates to keep notebook runtime sane.
top_for_redundancy = [r['feature'] for r in stat_signal_df.orderBy(F.desc('mutual_information')).limit(18).collect()]
redundancy_rows = []
for i, left in enumerate(top_for_redundancy):
    for right in top_for_redundancy[i + 1:]:
        corr = pearson_corr(feature_df.select(left, right).withColumn(TARGET_COL, F.col(right)), left, target=TARGET_COL)
        redundancy_rows.append({'feature_a': left, 'feature_b': right, 'pearson_corr': corr})

redundancy_df = spark.createDataFrame(redundancy_rows) if redundancy_rows else spark.createDataFrame([], 'feature_a string, feature_b string, pearson_corr double')
redundancy_df.where(F.abs(F.col('pearson_corr')) >= 0.85).orderBy(F.desc(F.abs('pearson_corr'))).show(80, truncate=False)


+-------------------------------+---------------------------------+------------------+
|feature_a                      |feature_b                        |pearson_corr      |
+-------------------------------+---------------------------------+------------------+
|session_prior_avg_watch_ratio  |session_vs_user_watch_ratio_delta|0.92199267528783  |
|session_vs_user_long_view_delta|session_prior_long_view_rate     |0.8773365638323076|
+-------------------------------+---------------------------------+------------------+



## 9. Temporal Stability

For each chronological split, we recompute target rate and feature MI. Stable features should not only work in one narrow time window.


In [12]:
top_for_stability = [r['feature'] for r in stat_signal_df.orderBy(F.desc('mutual_information')).limit(15).collect()]
stability_rows = []
for split in ['train', 'validation', 'test']:
    split_df = feature_df.where(F.col('split') == split)
    rows = split_df.count()
    target_rate = split_df.agg(F.avg(TARGET_COL)).first()[0]
    for feature in top_for_stability:
        stability_rows.append({
            'split': split,
            'feature': feature,
            'rows': rows,
            'target_rate': float(target_rate or 0.0),
            'mutual_information': binned_mi(split_df, feature),
        })

stability_df = spark.createDataFrame(stability_rows)
stability_df.orderBy('feature', 'split').show(120, truncate=False)

stability_summary = (
    stability_df.groupBy('feature')
    .agg(
        F.mean('mutual_information').alias('mi_mean'),
        F.stddev('mutual_information').alias('mi_stddev'),
        (F.stddev('mutual_information') / (F.mean('mutual_information') + F.lit(1e-9))).alias('mi_cv')
    )
)
stability_summary.orderBy(F.desc('mi_mean')).show(80, truncate=False)


+---------------------------------+---------------------+------+----------+------------------+
|feature                          |mutual_information   |rows  |split     |target_rate       |
+---------------------------------+---------------------+------+----------+------------------+
|aspect_ratio                     |0.0024135428549849477|35543 |test      |0.2550713220606026|
|aspect_ratio                     |0.0032363291941610957|164784|train     |0.2632355083017769|
|aspect_ratio                     |0.0034690472342649645|35331 |validation|0.2606492881605389|
|item_hist_long_view_rate         |0.010689365361380421 |35543 |test      |0.2550713220606026|
|item_hist_long_view_rate         |0.007916576596851644 |164784|train     |0.2632355083017769|
|item_hist_long_view_rate         |0.007409533377336207 |35331 |validation|0.2606492881605389|
|music_type                       |0.0022154540969830643|35543 |test      |0.2550713220606026|
|music_type                       |0.0033493161617

Interpretation: high `mi_cv` means the signal moves a lot over time. That does not always mean DROP, but it is a warning for adaptive monitoring and retraining cadence.


## 10. Segment Analysis

We examine light/heavy users, short/long sessions, high-drift sessions, and popular/tail items. High-drift sessions are especially important for the adaptive recommender.


In [13]:
segmented = (
    feature_df
    .withColumn('user_activity_segment', F.when(F.col('user_hist_events') >= 20, 'heavy').when(F.col('user_hist_events') >= 5, 'medium').otherwise('light'))
    .withColumn('session_length_segment', F.when(F.col('session_event_index') >= 10, 'long_session').when(F.col('session_event_index') >= 3, 'mid_session').otherwise('short_session'))
    .withColumn('drift_abs', F.abs(F.coalesce('session_vs_user_long_view_delta', F.lit(0.0))))
    .withColumn('drift_segment', F.when(F.col('drift_abs') >= 0.30, 'high_drift').when(F.col('drift_abs') >= 0.10, 'mid_drift').otherwise('low_drift'))
    .withColumn('item_popularity_segment', F.when(F.col('item_hist_events') >= 5, 'seen_in_sample').otherwise('tail_or_cold'))
)

for segment_col in ['user_activity_segment', 'session_length_segment', 'drift_segment', 'item_popularity_segment']:
    print(f'\n{segment_col}')
    segmented.groupBy(segment_col).agg(
        F.count('*').alias('rows'),
        F.avg(TARGET_COL).alias('target_rate'),
        F.avg('session_prior_long_view_rate').alias('avg_session_prior_long_view_rate'),
        F.avg('user_hist_long_view_rate').alias('avg_user_hist_long_view_rate'),
        F.avg('item_hist_long_view_rate').alias('avg_item_hist_long_view_rate'),
    ).orderBy(segment_col).show(truncate=False)



user_activity_segment


+---------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|user_activity_segment|rows  |target_rate        |avg_session_prior_long_view_rate|avg_user_hist_long_view_rate|avg_item_hist_long_view_rate|
+---------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|heavy                |215923|0.2536922884546806 |0.2235323658987971              |0.25700411920024685         |0.3488417519923249          |
|light                |4994  |0.35562675210252304|0.30454545454545456             |0.36550659322316814         |0.3777313827686963          |
|medium               |14741 |0.34583813852520184|0.30586085276304864             |0.3490567198558568          |0.39731467844869905         |
+---------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+


sess

+----------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|session_length_segment|rows  |target_rate        |avg_session_prior_long_view_rate|avg_user_hist_long_view_rate|avg_item_hist_long_view_rate|
+----------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|long_session          |13363 |0.08022150714659881|0.08569868876465196             |0.13012882390542294         |0.2927116348433313          |
|mid_session           |54659 |0.1766589216780402 |0.19049112433596238             |0.2128696727394644          |0.3276484708446734          |
|short_session         |167636|0.3037772316208929 |0.28178448504103787             |0.29239914517084453         |0.36125164637501184         |
+----------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+

+-------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|drift_segment|rows  |target_rate        |avg_session_prior_long_view_rate|avg_user_hist_long_view_rate|avg_item_hist_long_view_rate|
+-------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|high_drift   |38739 |0.35315831590903224|0.511680517233091               |0.36493084940456033         |0.37568190144294406         |
|low_drift    |136871|0.2672662580093665 |0.12251435303476113             |0.25650247520615194         |0.3566378314657737          |
|mid_drift    |60048 |0.18968158806288302|0.12549417514017613             |0.21832658684694467         |0.32175628929707234         |
+-------------+------+-------------------+--------------------------------+----------------------------+----------------------------+


item_popularity_segment


+-----------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|item_popularity_segment|rows  |target_rate        |avg_session_prior_long_view_rate|avg_user_hist_long_view_rate|avg_item_hist_long_view_rate|
+-----------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+
|seen_in_sample         |537   |0.41527001862197394|0.32966735364582256             |0.33468443368673495         |0.42270201348413644         |
|tail_or_cold           |235121|0.26126547607402145|0.2285803137813536              |0.2644740718319192          |0.3505308362161519          |
+-----------------------+------+-------------------+--------------------------------+----------------------------+----------------------------+



In [14]:
segment_mi_features = [f for f in ['session_prior_long_view_rate', 'user_hist_long_view_rate', 'session_vs_user_long_view_delta', 'item_hist_long_view_rate', 'video_duration_sec'] if f in safe_numeric_features]
segment_mi_rows = []
for segment in ['low_drift', 'mid_drift', 'high_drift']:
    sdf = segmented.where(F.col('drift_segment') == segment)
    if sdf.count() < 100:
        continue
    for feature in segment_mi_features:
        segment_mi_rows.append({'segment': segment, 'feature': feature, 'mutual_information': binned_mi(sdf, feature)})
segment_mi_df = spark.createDataFrame(segment_mi_rows)
segment_mi_df.orderBy('segment', F.desc('mutual_information')).show(80, truncate=False)


+-------------------------------+---------------------+----------+
|feature                        |mutual_information   |segment   |
+-------------------------------+---------------------+----------+
|user_hist_long_view_rate       |0.024032864405427727 |high_drift|
|session_vs_user_long_view_delta|0.00959803047264007  |high_drift|
|session_prior_long_view_rate   |0.007168748558429418 |high_drift|
|video_duration_sec             |0.0056834143723461655|high_drift|
|item_hist_long_view_rate       |0.005173133886633395 |high_drift|
|user_hist_long_view_rate       |0.06990625879356095  |low_drift |
|session_prior_long_view_rate   |0.0472728406559776   |low_drift |
|session_vs_user_long_view_delta|0.009895476072158842 |low_drift |
|item_hist_long_view_rate       |0.009557245191980965 |low_drift |
|video_duration_sec             |0.0036363689432143   |low_drift |
|user_hist_long_view_rate       |0.026792307923798837 |mid_drift |
|session_prior_long_view_rate   |0.0181192988859502   |mid_dri

Interpretation: if session/drift features gain relative MI in `high_drift`, they are strong candidates for the adaptive component even when their global score is only moderate.


## 11. Model-Based Feature Importance

We use Spark ML on a temporal split. This is not the final recommender model; it is a tabular proxy for feature usefulness with a binary long-view target.

Limitation: the target is `long_view`, not a full ranking objective with negatives/candidates. This is acceptable for feature screening, not final offline ranking claims.


In [15]:
# Do not include high-leakage post-event features or unknown-snapshot global statistics in the default model set.
high_leakage_features = [r['feature'] for r in catalog_rows if r['leakage_risk'] == 'HIGH']
model_numeric_features = [
    c for c in safe_numeric_features
    if c not in high_leakage_features and c in feature_df.columns
]
# Keep the model compact and interpretable.
priority_order = [
    'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate', 'user_hist_hate_rate', 'user_hist_avg_watch_ratio',
    'follow_user_num', 'fans_user_num', 'register_days',
    'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate', 'session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta',
    'item_hist_events', 'item_hist_long_view_rate',
    'video_duration_sec', 'aspect_ratio',
    'event_hour', 'event_dayofweek', 'is_rand', 'tab',
]
model_numeric_features = [c for c in priority_order if c in model_numeric_features]
print('Model features:', model_numeric_features)

model_df = (
    feature_df
    .select('split', TARGET_COL, *model_numeric_features)
    .limit(MAX_MODEL_ROWS)
    .cache()
)
model_df.groupBy('split').agg(F.count('*').alias('rows'), F.avg(TARGET_COL).alias('target_rate')).orderBy('split').show()

imputer = Imputer(inputCols=model_numeric_features, outputCols=[f'{c}__imputed' for c in model_numeric_features]).setStrategy('median')
assembler = VectorAssembler(inputCols=[f'{c}__imputed' for c in model_numeric_features], outputCol='features')
rf = RandomForestClassifier(labelCol=TARGET_COL, featuresCol='features', numTrees=40, maxDepth=7, seed=SEED, subsamplingRate=0.8)
pipeline = Pipeline(stages=[imputer, assembler, rf])

train_df = model_df.where(F.col('split') == 'train')
validation_df = model_df.where(F.col('split') == 'validation')
test_df = model_df.where(F.col('split') == 'test')

evaluator = BinaryClassificationEvaluator(labelCol=TARGET_COL, rawPredictionCol='rawPrediction', metricName='areaUnderROC')
rf_model = pipeline.fit(train_df)
val_auc = evaluator.evaluate(rf_model.transform(validation_df))
test_auc = evaluator.evaluate(rf_model.transform(test_df))
print(f'RandomForest validation AUC={val_auc:.4f}, test AUC={test_auc:.4f}')

rf_stage = rf_model.stages[-1]
importance_rows = [
    {'feature': feature, 'rf_importance': float(rf_stage.featureImportances[idx])}
    for idx, feature in enumerate(model_numeric_features)
]
model_importance_df = spark.createDataFrame(importance_rows)
model_importance_df.orderBy(F.desc('rf_importance')).show(80, truncate=False)


Model features: ['user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate', 'user_hist_hate_rate', 'user_hist_avg_watch_ratio', 'follow_user_num', 'fans_user_num', 'register_days', 'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate', 'session_prior_avg_watch_ratio', 'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta', 'item_hist_events', 'item_hist_long_view_rate', 'video_duration_sec', 'aspect_ratio', 'event_hour', 'event_dayofweek', 'is_rand', 'tab']


+----------+------+------------------+
|     split|  rows|       target_rate|
+----------+------+------------------+
|      test| 35543|0.2550713220606026|
|     train|164784|0.2632355083017769|
|validation| 35331|0.2606492881605389|
+----------+------+------------------+



26/09/23 17:04:13 WARN DAGScheduler: Broadcasting large task binary with size 1034.4 KiB


RandomForest validation AUC=0.7673, test AUC=0.7654
+---------------------------------+---------------------+
|feature                          |rf_importance        |
+---------------------------------+---------------------+
|user_hist_long_view_rate         |0.28183135090257433  |
|tab                              |0.2724706118058405   |
|user_hist_avg_watch_ratio        |0.20998390370020306  |
|session_event_index              |0.04609000104953189  |
|session_prior_avg_watch_ratio    |0.03372231047080679  |
|video_duration_sec               |0.032003691144926245 |
|session_vs_user_long_view_delta  |0.029269213874778273 |
|user_hist_events                 |0.024824687882262313 |
|session_elapsed_sec              |0.020204723528394527 |
|session_prior_long_view_rate     |0.01872743014469283  |
|session_vs_user_watch_ratio_delta|0.011285072895924692 |
|item_hist_long_view_rate         |0.003973465351817874 |
|fans_user_num                    |0.003756056482172218 |
|aspect_ratio       

In [16]:
# Approximate permutation importance: replace one feature with its train median and measure test AUC drop.
# This is cheaper and more reproducible in Spark than collecting a shuffled pandas dataset.
train_medians = train_df.approxQuantile(model_numeric_features, [0.5], 0.01)
median_map = {feature: float(values[0]) if values else 0.0 for feature, values in zip(model_numeric_features, train_medians)}
baseline_auc = test_auc
perm_rows = []
for feature in [r['feature'] for r in model_importance_df.orderBy(F.desc('rf_importance')).limit(10).collect()]:
    perturbed = test_df.withColumn(feature, F.lit(median_map.get(feature, 0.0)))
    auc = evaluator.evaluate(rf_model.transform(perturbed))
    perm_rows.append({'feature': feature, 'test_auc_when_replaced_by_train_median': float(auc), 'auc_drop': float(baseline_auc - auc)})

permutation_df = spark.createDataFrame(perm_rows)
permutation_df.orderBy(F.desc('auc_drop')).show(40, truncate=False)


+---------------------+-------------------------------+--------------------------------------+
|auc_drop             |feature                        |test_auc_when_replaced_by_train_median|
+---------------------+-------------------------------+--------------------------------------+
|0.03186062799190681  |tab                            |0.7335125435217216                    |
|0.00932852234482684  |user_hist_long_view_rate       |0.7560446491688015                    |
|0.004064193638804836 |video_duration_sec             |0.7613089778748235                    |
|0.001153240893759011 |session_prior_long_view_rate   |0.7642199306198694                    |
|9.897663844884441E-4 |session_event_index            |0.7643834051291399                    |
|8.620379290857905E-4 |session_elapsed_sec            |0.7645111335845426                    |
|3.038529142764812E-4 |session_prior_avg_watch_ratio  |0.7650693185993519                    |
|5.40033909780524E-5  |user_hist_events           

## 12. Feature-Group Ablation

Ablation is cumulative and chronological:

1. item features only
2. + long-term user features
3. + session features
4. + drift/adaptation features
5. + context features

The point is directionality, not squeezing maximum AUC from this proxy model.


In [17]:
group_map = {r['feature']: r['group'] for r in catalog_rows}
feature_groups = {
    'item_only': [f for f in model_numeric_features if group_map.get(f) == 'item'],
    'plus_long_term_user': [f for f in model_numeric_features if group_map.get(f) in ['item', 'long_term_user']],
    'plus_session': [f for f in model_numeric_features if group_map.get(f) in ['item', 'long_term_user', 'session']],
    'plus_drift': [f for f in model_numeric_features if group_map.get(f) in ['item', 'long_term_user', 'session', 'drift/adaptation']],
    'plus_context': [f for f in model_numeric_features if group_map.get(f) in ['item', 'long_term_user', 'session', 'drift/adaptation', 'context']],
}

ablation_rows = []
for name, features in feature_groups.items():
    if not features:
        continue
    imputer_i = Imputer(inputCols=features, outputCols=[f'{c}__imp' for c in features]).setStrategy('median')
    assembler_i = VectorAssembler(inputCols=[f'{c}__imp' for c in features], outputCol='features')
    lr = LogisticRegression(labelCol=TARGET_COL, featuresCol='features', maxIter=25, regParam=0.05, elasticNetParam=0.0)
    pipe = Pipeline(stages=[imputer_i, assembler_i, lr])
    model = pipe.fit(train_df.select('split', TARGET_COL, *features))
    val_auc_i = evaluator.evaluate(model.transform(validation_df.select('split', TARGET_COL, *features)))
    test_auc_i = evaluator.evaluate(model.transform(test_df.select('split', TARGET_COL, *features)))
    ablation_rows.append({'ablation_step': name, 'num_features': len(features), 'features': ', '.join(features), 'validation_auc': float(val_auc_i), 'test_auc': float(test_auc_i)})

ablation_df = spark.createDataFrame(ablation_rows)
ablation_df.orderBy('num_features').show(truncate=False)


26/09/23 17:04:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/09/23 17:04:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


+-------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------+------------------+
|ablation_step      |features                                                                                                                                                                                                                                                                                                                                                                                                                                                          |num_

Interpretation: if session/drift/context additions improve temporal validation/test AUC, they deserve gold-layer support. If they do not move this proxy metric, still inspect high-drift segment evidence before dropping them.


## 13. Feature Selection Summary

The final decision combines leakage risk, quality, MI, redundancy, stability, model importance, ablation contribution, and online cost. This is intentionally conservative but not overly aggressive: silver can keep rich data, while gold V0 should start with a focused set.


In [18]:
# Join evidence into a compact summary table.
quality_small = quality_df.select('feature', 'missing_rate', 'coverage', 'approx_cardinality', 'near_zero_variance')
stat_small = stat_signal_df.select('feature', 'pearson', 'spearman_proxy', 'mutual_information')
stability_small = stability_summary.select('feature', 'mi_mean', 'mi_cv')
importance_small = model_importance_df.select('feature', 'rf_importance')
perm_small = permutation_df.select('feature', 'auc_drop')

summary = (
    feature_catalog
    .join(quality_small, 'feature', 'left')
    .join(stat_small, 'feature', 'left')
    .join(stability_small, 'feature', 'left')
    .join(importance_small, 'feature', 'left')
    .join(perm_small, 'feature', 'left')
    .withColumn(
        'online_computation_cost',
        F.when(F.col('group').isin('session', 'drift/adaptation'), F.lit('MEDIUM'))
         .when(F.col('group') == 'item', F.lit('LOW/MEDIUM'))
         .otherwise(F.lit('LOW'))
    )
    .withColumn(
        'final_decision',
        F.when(F.col('leakage_risk') == 'HIGH', F.lit('DROP'))
         .when(F.col('exists_in_current_data') == False, F.lit('DROP'))
         .when(F.col('near_zero_variance') == True, F.lit('DROP'))
         .when((F.coalesce(F.col('rf_importance'), F.lit(0.0)) >= 0.02) | (F.coalesce(F.col('auc_drop'), F.lit(0.0)) >= 0.001), F.lit('KEEP'))
         .when(F.col('group').isin('session', 'drift/adaptation') & (F.coalesce(F.col('mutual_information'), F.lit(0.0)) > 0.0001), F.lit('KEEP'))
         .when(F.coalesce(F.col('mutual_information'), F.lit(0.0)) > 0.00005, F.lit('OPTIONAL'))
         .otherwise(F.lit('OPTIONAL'))
    )
    .withColumn(
        'short_reason',
        F.when(F.col('leakage_risk') == 'HIGH', F.lit('Not available at pre-ranking prediction time or snapshot may contain future interactions.'))
         .when(F.col('final_decision') == 'KEEP', F.lit('Point-in-time safe and supported by statistical/model or adaptive-session evidence.'))
         .when(F.col('final_decision') == 'OPTIONAL', F.lit('Keep for experiments or later gold versions; evidence is weaker or serving value is context dependent.'))
         .otherwise(F.lit('Low evidence or unsuitable for V0.'))
    )
    .select(
        'feature', 'group', 'source_columns', 'derivation',
        'mutual_information', 'pearson', 'spearman_proxy',
        'missing_rate', 'coverage', 'approx_cardinality',
        'mi_cv', 'rf_importance', 'auc_drop',
        'online_computation_cost', 'leakage_risk', 'final_decision', 'short_reason'
    )
)

summary.orderBy('final_decision', 'group', F.desc('rf_importance')).show(120, truncate=False)


+---------------------------------+----------------+-----------------------------------------+-------------------------------------------------------------+---------------------+---------------------+---------------------+--------------------+-------------------+------------------+--------------------+---------------------+---------------------+-----------------------+------------+--------------+------------------------------------------------------------------------------------------------------+
|feature                          |group           |source_columns                           |derivation                                                   |mutual_information   |pearson              |spearman_proxy       |missing_rate        |coverage           |approx_cardinality|mi_cv               |rf_importance        |auc_drop             |online_computation_cost|leakage_risk|final_decision|short_reason                                                                                      

In [19]:
keep_features = [r['feature'] for r in summary.where(F.col('final_decision') == 'KEEP').select('feature').collect()]
optional_features = [r['feature'] for r in summary.where(F.col('final_decision') == 'OPTIONAL').select('feature').collect()]
drop_features = [r['feature'] for r in summary.where(F.col('final_decision') == 'DROP').select('feature').collect()]

minimum_v0 = [f for f in [
    'video_duration_sec', 'aspect_ratio', 'item_hist_events', 'item_hist_long_view_rate',
    'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate', 'user_hist_hate_rate',
    'session_event_index', 'session_prior_long_view_rate',
    'session_vs_user_long_view_delta', 'event_hour', 'event_dayofweek', 'tab'
] if f in keep_features or f in optional_features]

recommended_main = [f for f in [
    'video_duration_sec', 'aspect_ratio', 'item_hist_events', 'item_hist_long_view_rate',
    'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate', 'user_hist_hate_rate', 'user_hist_avg_watch_ratio',
    'follow_user_num', 'fans_user_num', 'register_days',
    'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate', 'session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta',
    'event_hour', 'event_dayofweek', 'tab', 'is_rand'
] if f in keep_features or f in optional_features]

future_experiments = [f for f in optional_features if f not in recommended_main]

print('A. Minimum feature set for Adaptive Recommender V0')
for f in minimum_v0:
    print('-', f)

print('\nB. Recommended feature set for the main ranking model')
for f in recommended_main:
    print('-', f)

print('\nC. Features to keep for future experiments')
for f in future_experiments:
    print('-', f)

print('\nFeatures to exclude from pre-ranking model unless rebuilt point-in-time')
for f in drop_features:
    print('-', f)


A. Minimum feature set for Adaptive Recommender V0
- video_duration_sec
- aspect_ratio
- item_hist_events
- item_hist_long_view_rate
- user_hist_events
- user_hist_long_view_rate
- user_hist_like_rate
- user_hist_hate_rate
- session_event_index
- session_prior_long_view_rate
- session_vs_user_long_view_delta
- event_hour
- event_dayofweek
- tab

B. Recommended feature set for the main ranking model
- video_duration_sec
- aspect_ratio
- item_hist_events
- item_hist_long_view_rate
- user_hist_events
- user_hist_long_view_rate
- user_hist_like_rate
- user_hist_hate_rate
- user_hist_avg_watch_ratio
- follow_user_num
- fans_user_num
- register_days
- session_event_index
- session_elapsed_sec
- session_prior_long_view_rate
- session_prior_avg_watch_ratio
- session_vs_user_long_view_delta
- session_vs_user_watch_ratio_delta
- event_hour
- event_dayofweek
- tab
- is_rand

C. Features to keep for future experiments
- user_active_degree
- video_type
- music_type

Features to exclude from pre-ran

## 14. Gold Design Notes

Recommended next implementation step:

- Build a gold training table with one row per candidate impression/event.
- Use temporal train/validation/test splits only.
- Materialize point-in-time `user_hist_*`, `item_hist_*`, `session_*`, and `drift_*` fields.
- Keep raw silver tables rich; apply feature selection in gold/model configs.
- Treat `videos_statistics` aggregate rates as unsafe for final offline claims until their snapshot time is known or they are rebuilt as historical windows before each prediction timestamp.
